In [ ]:
# =============================================================================
# EXTRACTION AUTOMATISÉE ERA5 + CHIRPS DEPUIS GOOGLE EARTH ENGINE
# Google Colab — copier-coller ce script dans une cellule unique ou en blocs
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# BLOC 1 — Installation & imports
# ─────────────────────────────────────────────────────────────────────────────
!pip install earthengine-api --quiet

import ee
import pandas as pd
import numpy as np
import io
import os
from datetime import datetime
from google.colab import files

print("✅ Librairies chargées.")


# ─────────────────────────────────────────────────────────────────────────────
# BLOC 2 — Authentification Google Earth Engine
# Un lien s'ouvre → copier le token → coller ici
# ─────────────────────────────────────────────────────────────────────────────
ee.Authenticate()

# ⚠️  ADAPTER : remplacer par l'ID de votre projet GEE
GEE_PROJECT = 'earth-engine-project-489915'
ee.Initialize(project=GEE_PROJECT)

print(f"✅ GEE initialisé — projet : {GEE_PROJECT}")


# ─────────────────────────────────────────────────────────────────────────────
# BLOC 3 — Chargement du fichier stations_summary.csv
# Colonnes attendues : nom, lat, lon, depuis
# ─────────────────────────────────────────────────────────────────────────────
print("📂 Téléchargez votre fichier stations_summary.csv :")
uploaded = files.upload()

if not uploaded:
    raise FileNotFoundError("❌ Aucun fichier téléchargé. Relancez ce bloc.")

filename = list(uploaded.keys())[0]
df_stations = pd.read_csv(io.StringIO(uploaded[filename].decode('utf-8')))

# ── Validation des colonnes minimales ────────────────────────────────────────
COLS_REQUISES = {'nom', 'lat', 'lon', 'depuis'}
cols_csv = set(df_stations.columns.str.strip().str.lower())
manquantes = COLS_REQUISES - cols_csv
if manquantes:
    raise ValueError(f"❌ Colonnes manquantes dans le CSV : {manquantes}\n"
                     f"   Colonnes trouvées : {list(df_stations.columns)}")

# Normalisation des noms de colonnes (minuscules, sans espaces)
df_stations.columns = df_stations.columns.str.strip().str.lower()

# ── Construction du dictionnaire REGIONS ─────────────────────────────────────
REGIONS = {}
for i, row in df_stations.iterrows():
    rid = f'R{i+1:02d}'
    REGIONS[rid] = {
        'nom'   : str(row['nom']).strip(),
        'lat'   : float(row['lat']),
        'lon'   : float(row['lon']),
        'depuis': int(row['depuis']),
    }

print(f"\n✅ {len(REGIONS)} station(s) chargée(s) :\n")
for rid, info in REGIONS.items():
    print(f"  {rid} | {info['nom']:20s} | lat={info['lat']:.4f}  "
          f"lon={info['lon']:.4f}  depuis={info['depuis']}")


# ─────────────────────────────────────────────────────────────────────────────
# BLOC 4 — Paramètres d'extraction
# ─────────────────────────────────────────────────────────────────────────────

ANNEE_FIN = 2025        # Année de fin (inclusive)
SCALE_M   = 1000        # Résolution spatiale pour reduceRegion (mètres)

# Bandes ERA5-Land à extraire (moyennes mensuelles)
ERA5_BANDS = [
    'temperature_2m',
    'dewpoint_temperature_2m',
    'total_precipitation',
    'u_component_of_wind_10m',
    'v_component_of_wind_10m',
    'surface_solar_radiation_downwards',
    'surface_pressure',
]

print(f"⚙️  Extraction ERA5 + CHIRPS — jusqu'à {ANNEE_FIN}")
print(f"   Bandes ERA5 : {ERA5_BANDS}")


# ─────────────────────────────────────────────────────────────────────────────
# BLOC 5 — Fonction d'extraction mensuelle pour un point GPS
# ─────────────────────────────────────────────────────────────────────────────

def extraire_mensuel_gee(region_id, lat, lon, annee_debut, annee_fin=ANNEE_FIN):
    """
    Extrait ERA5-Land (moyenne) et CHIRPS (somme) mensuels pour un point GPS.
    Retourne un DataFrame avec une ligne par mois.

    Paramètres
    ----------
    region_id   : identifiant de la station (str)
    lat, lon    : coordonnées GPS (float)
    annee_debut : première année à extraire (int)
    annee_fin   : dernière année inclusive (int)

    Retour
    ------
    pd.DataFrame  colonnes : datetime, region_id, era5_*, chirps_precipitation
    """
    point      = ee.Geometry.Point([lon, lat])
    col_era5   = ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY')
    col_chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')

    rows = []
    annees = range(annee_debut, annee_fin + 1)
    total  = len(annees) * 12

    for i_y, y in enumerate(annees):
        for m in range(1, 13):
            pct = ((i_y * 12 + m) / total) * 100
            print(f"\r   {region_id} — {y}-{m:02d}  ({pct:.0f}%)", end='', flush=True)

            try:
                start = ee.Date.fromYMD(y, m, 1)
                end   = start.advance(1, 'month')

                # ERA5-Land : moyenne mensuelle des valeurs horaires
                era5_img = (col_era5
                            .filterDate(start, end)
                            .filterBounds(point)
                            .select(ERA5_BANDS)
                            .mean())
                era5_v = era5_img.reduceRegion(
                    reducer  = ee.Reducer.mean(),
                    geometry = point,
                    scale    = SCALE_M,
                    bestEffort = True,
                ).getInfo()

                # CHIRPS : somme mensuelle des précipitations journalières
                chirps_v = (col_chirps
                            .filterDate(start, end)
                            .filterBounds(point)
                            .select(['precipitation'])
                            .sum()
                            .reduceRegion(
                                reducer  = ee.Reducer.mean(),
                                geometry = point,
                                scale    = SCALE_M,
                                bestEffort = True,
                            ).getInfo())

                row = {
                    'datetime' : pd.Timestamp(y, m, 1),
                    'region_id': region_id,
                }
                row.update({f'era5_{k}': v for k, v in era5_v.items()})
                row.update({f'chirps_{k}': v for k, v in chirps_v.items()})
                rows.append(row)

            except Exception as e:
                # Mois sans données (ex. : CHIRPS non disponible avant 1981)
                print(f"\n   ⚠️  {region_id} {y}-{m:02d} ignoré : {e}")

    print()  # saut de ligne après la barre de progression
    return pd.DataFrame(rows)


# ─────────────────────────────────────────────────────────────────────────────
# BLOC 6 — Lancement de l'extraction pour toutes les stations
# ─────────────────────────────────────────────────────────────────────────────

dfs = []

for rid, info in REGIONS.items():
    print(f"\n🛰  Extraction GEE — {rid} : {info['nom']} "
          f"(lat={info['lat']}, lon={info['lon']}, depuis={info['depuis']})")

    df_region = extraire_mensuel_gee(
        region_id   = rid,
        lat         = info['lat'],
        lon         = info['lon'],
        annee_debut = info['depuis'],
        annee_fin   = ANNEE_FIN,
    )

    # Ajout des métadonnées de station
    df_region['region_nom'] = info['nom']
    df_region['lat']        = info['lat']
    df_region['lon']        = info['lon']

    dfs.append(df_region)
    print(f"   ✅ {len(df_region)} mois extraits pour {info['nom']}")

df_final = pd.concat(dfs, ignore_index=True)
print(f"\n✅ Extraction terminée — {len(df_final)} lignes au total.")


# ─────────────────────────────────────────────────────────────────────────────
# BLOC 7 — Post-traitement : conversions d'unités ERA5
# ─────────────────────────────────────────────────────────────────────────────

# Température : Kelvin → Celsius
for col in ['era5_temperature_2m', 'era5_dewpoint_temperature_2m']:
    if col in df_final.columns:
        df_final[col.replace('era5_', '') + '_C'] = df_final[col] - 273.15

# Précipitations ERA5 : m → mm  (total_precipitation est en mètres dans ERA5)
if 'era5_total_precipitation' in df_final.columns:
    df_final['era5_precip_mm'] = df_final['era5_total_precipitation'] * 1000

# Vitesse du vent scalaire (m/s)
if {'era5_u_component_of_wind_10m', 'era5_v_component_of_wind_10m'}.issubset(df_final.columns):
    df_final['era5_wind_speed_ms'] = np.sqrt(
        df_final['era5_u_component_of_wind_10m']**2 +
        df_final['era5_v_component_of_wind_10m']**2
    )

# Rayonnement : J/m² → MJ/m²
if 'era5_surface_solar_radiation_downwards' in df_final.columns:
    df_final['era5_radiation_MJm2'] = (
        df_final['era5_surface_solar_radiation_downwards'] / 1e6
    )

# Pression : Pa → hPa
if 'era5_surface_pressure' in df_final.columns:
    df_final['era5_pressure_hPa'] = df_final['era5_surface_pressure'] / 100

print("✅ Conversions d'unités appliquées.")
print(df_final.head())


# ─────────────────────────────────────────────────────────────────────────────
# BLOC 8 — Sauvegarde et téléchargement
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_CSV = f'extraction_gee_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'

df_final.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f"💾 Fichier sauvegardé : {OUTPUT_CSV}")
print(f"   {df_final.shape[0]} lignes × {df_final.shape[1]} colonnes")

# Téléchargement automatique dans le navigateur
files.download(OUTPUT_CSV)
print("⬇️  Téléchargement lancé.")


# ─────────────────────────────────────────────────────────────────────────────
# BLOC 9 — Aperçu statistique rapide
# ─────────────────────────────────────────────────────────────────────────────

print("\n📊 Résumé par station :")
print(
    df_final.groupby('region_nom')[
        [c for c in df_final.columns if c.startswith('era5_') or c.startswith('chirps_')]
    ].describe().round(2).T
)

print("\n📅 Plage temporelle :")
print(df_final.groupby('region_nom')['datetime'].agg(['min', 'max']))

✅ Librairies chargées.
✅ GEE initialisé — projet : earth-engine-project-489915
📂 Téléchargez votre fichier stations_summary.csv :


Saving stations_summary.csv to stations_summary.csv

✅ 24 station(s) chargée(s) :

  R01 | ksar El Kebir        | lat=35.0035  lon=-5.9155  depuis=2012
  R02 | DAR ELGUEDDARI       | lat=34.4113  lon=-6.1101  depuis=2012
  R03 | SIDI ALLAL TAZI      | lat=34.5220  lon=-6.3428  depuis=2015
  R04 | SUTA - FERME ABT     | lat=32.2123  lon=-6.7878  depuis=2015
  R05 | SUTA-TAZEROUALT      | lat=32.2168  lon=-6.7972  depuis=2016
  R06 | SUTA OULAD AYAD      | lat=32.2123  lon=-6.7878  depuis=2015
  R07 | MECHRAA BELKSIRI     | lat=34.5740  lon=-5.9515  depuis=2018
  R08 | SUTA 507             | lat=32.4356  lon=-6.7510  depuis=2019
  R09 | M-Zaio               | lat=34.9275  lon=-2.7537  depuis=2016
  R10 | Merksen              | lat=32.4357  lon=-6.7510  depuis=2017
  R11 | SUTA_CENTAGRI        | lat=32.3411  lon=-6.9802  depuis=2017
  R12 | SUTA INRA            | lat=32.2604  lon=-6.5250  depuis=2018
  R13 | SUTA 503             | lat=32.5127  lon=-6.5726  depuis=2018
  R14 | SUTA 505    

   R03 — 2025-12  (100%)
   ✅ 132 mois extraits pour SIDI ALLAL TAZI

🛰  Extraction GEE — R04 : SUTA - FERME ABT (lat=32.212323853775, lon=-6.787755302734, depuis=2015)
   R04 — 2017-04  (21%)

   R04 — 2025-12  (100%)
   ✅ 132 mois extraits pour SUTA - FERME ABT

🛰  Extraction GEE — R05 : SUTA-TAZEROUALT (lat=32.21676586394668, lon=-6.797247314453102, depuis=2016)
   R05 — 2025-12  (100%)
   ✅ 120 mois extraits pour SUTA-TAZEROUALT

🛰  Extraction GEE — R06 : SUTA OULAD AYAD (lat=32.21232385377554, lon=-6.78775530273424, depuis=2015)
   R06 — 2025-12  (100%)
   ✅ 132 mois extraits pour SUTA OULAD AYAD

🛰  Extraction GEE — R07 : MECHRAA BELKSIRI (lat=34.5739933, lon=-5.951539899999943, depuis=2018)
   R07 — 2025-12  (100%)
   ✅ 96 mois extraits pour MECHRAA BELKSIRI

🛰  Extraction GEE — R08 : SUTA 507 (lat=32.435584, lon=-6.75096, depuis=2019)
   R08 — 2025-12  (100%)
   ✅ 84 mois extraits pour SUTA 507

🛰  Extraction GEE — R09 : M-Zaio (lat=34.927492, lon=-2.753676, depuis=2016)
   R09 — 2025-12  (100%)
   ✅ 120 mois extraits pour M-Zaio

🛰  Extraction GEE — R10 : Merksen (lat=32.435671, lon=-6.750984, depuis=2017)
   R10 — 2025-12  (100%)
   ✅ 108 mois extraits pour Merksen
